# Data Preparation and Consolidation — United States

##  Objective

The objective of this stage is to inspect, clean, and consolidate the US bilateral trade dataset into a unified monthly dataset suitable for subsequent exploratory data analysis and forecasting.

The processing workflow focuses on:

* Verifying the structure and temporal coverage of the source data.
* Examining data completeness and consistency.
* Identifying and investigating duplicate observations.
* Establishing an appropriate procedure for handling duplicate records.
* Consolidating observations at the intended analytical level.
* Exporting the processed dataset in Parquet format.
* Validating the resulting dataset.

The original source file will be retained without modification.

## 1. Imports and Project Configuration

In [2]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
con = duckdb.connect(database=":memory:")

In [3]:
# Project directories
PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "Data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "Data" / "processed"

# US trade data
USA_FILE = RAW_DIR / "trade_s_usa_state_m_hs.csv"

# Create processed directory if it does not exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Display paths and input file information
print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

assert RAW_DIR.exists(), "Raw data directory does not exist."
assert PROCESSED_DIR.exists(), "Processed data directory does not exist."
assert USA_FILE.exists(), "US source file does not exist."

print(f"\nUS source file: {USA_FILE.name}")
print(f"File size: {USA_FILE.stat().st_size / (1024**3):.2f} GB")

Project directory: c:\Transfer\InterdisciplinaryProject
Raw data directory: c:\Transfer\InterdisciplinaryProject\data\raw
Processed data directory: c:\Transfer\InterdisciplinaryProject\data\processed

US source file: trade_s_usa_state_m_hs.csv
File size: 43.65 GB


## 2. Data Source and Input File

The US trade data are provided in a single CSV file containing monthly bilateral trade observations at the state and product levels.

The original source file is stored in `data/raw/` and will be retained without modification.

### Original schema

The source file will be inspected to identify its column names, data types, and structure before defining the subsequent data-cleaning and consolidation procedures.

In [3]:
schema = con.execute(
    "DESCRIBE SELECT * FROM read_csv_auto(?)",
    [str(USA_FILE)]
).df()

display(schema)

,column_name,column_type,null,key,default,extra
0,month_id,BIGINT,YES,None,None,None
1,state_id,VARCHAR,YES,None,None,None
2,state_name,VARCHAR,YES,None,None,None
3,trade_flow_name,VARCHAR,YES,None,None,None
4,country_id,VARCHAR,YES,None,None,None
5,country_name,VARCHAR,YES,None,None,None
6,product_id,VARCHAR,YES,None,None,None
7,product_name,VARCHAR,YES,None,None,None
8,trade_value,BIGINT,YES,None,None,None


## 3. Data Inspection and Quality Assessment

### 3.1 Temporal coverage

The dataset will be examined to identify its first and last observation periods and the number of distinct months.

In [4]:
temporal_coverage = con.execute("""
    SELECT
        MIN(month_id) AS first_month,
        MAX(month_id) AS last_month,
        COUNT(DISTINCT month_id) AS distinct_months
    FROM read_csv_auto(?)
""", [str(USA_FILE)]).df()

display(temporal_coverage.T)

,0
first_month,201701
last_month,202604
distinct_months,112


### 3.2 Temporal Consistency Check
This section compares monthly identifiers against the expected sequence of consecutive months to identify any missing or unexpected reporting periods.

In [7]:
temporal_consistency = con.execute("""
    WITH observed_months AS (
        SELECT DISTINCT month_id
        FROM read_csv_auto(?)
    ),
    expected_months AS (
        SELECT
            year * 100 + month AS month_id
        FROM range(2017, 2027) AS years(year)
        CROSS JOIN range(1, 13) AS months(month)
        WHERE year * 100 + month BETWEEN 201701 AND 202604
    )
    SELECT
        (SELECT COUNT(*) FROM observed_months) AS observed_months,
        (SELECT COUNT(*) FROM expected_months) AS expected_months,
        (
            SELECT COUNT(*)
            FROM expected_months e
            LEFT JOIN observed_months o USING (month_id)
            WHERE o.month_id IS NULL
        ) AS missing_months,
        (
            SELECT COUNT(*)
            FROM observed_months o
            LEFT JOIN expected_months e USING (month_id)
            WHERE e.month_id IS NULL
        ) AS unexpected_months
    FROM (SELECT 1)
""", [str(USA_FILE)]).df()

display(temporal_consistency.T)

,0
observed_months,112
expected_months,112
missing_months,0
unexpected_months,0


###  3.3 Initial Data Completeness
Check for null values in the core analytical variables, following the same approach as the Korea and Slovakia notebooks. For the US dataset, these are month_id, product_id, country_id, trade_flow_name, and trade_value. We'll also check state_id, since the state identifier is essential to this dataset's granularity.

In [8]:
null_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE month_id IS NULL) AS missing_month_id,
        COUNT(*) FILTER (WHERE state_id IS NULL) AS missing_state_id,
        COUNT(*) FILTER (WHERE product_id IS NULL) AS missing_product_id,
        COUNT(*) FILTER (WHERE country_id IS NULL) AS missing_country_id,
        COUNT(*) FILTER (WHERE trade_flow_name IS NULL) AS missing_trade_flow,
        COUNT(*) FILTER (WHERE trade_value IS NULL) AS missing_trade_value
    FROM read_csv_auto(?)
""", [str(USA_FILE)]).df()

display(null_check.T)

,0
total_rows,401741648
missing_month_id,0
missing_state_id,0
missing_product_id,0
missing_country_id,0
missing_trade_flow,0
missing_trade_value,0


### 3.4 Initial Dataset Statistics

In [9]:
initial_stats = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT month_id) AS distinct_months,
        COUNT(DISTINCT state_id) AS distinct_states,
        COUNT(DISTINCT product_id) AS distinct_products,
        COUNT(DISTINCT country_id) AS distinct_partners,
        COUNT(DISTINCT trade_flow_name) AS distinct_trade_flows,
        MIN(month_id) AS first_month,
        MAX(month_id) AS last_month
    FROM read_csv_auto(?)
""", [str(USA_FILE)]).df()

display(initial_stats.T)

,0
total_rows,401741648
distinct_months,112
distinct_states,54
distinct_products,5784
distinct_partners,231
distinct_trade_flows,2
first_month,201701
last_month,202604


## 4. Duplicate Investigation

Duplicate observations will be investigated using a proposed analytical key that accounts for the monthly period, state, product, trading partner, and trade flow.

The purpose is to determine whether repeated records represent exact repetitions or distinct observations that require further investigation before consolidation.

In [11]:
duplicate_key_check = con.execute("""
    WITH key_counts AS (
        SELECT
            month_id,
            state_id,
            product_id,
            country_id,
            trade_flow_name,
            COUNT(*) AS record_count
        FROM read_csv_auto(?)
        GROUP BY
            month_id,
            state_id,
            product_id,
            country_id,
            trade_flow_name
    )
    SELECT
        COUNT(*) FILTER (
            WHERE record_count > 1
        ) AS duplicated_key_groups,
        SUM(
            CASE
                WHEN record_count > 1 THEN record_count - 1
                ELSE 0
            END
        ) AS excess_records,
        MAX(record_count) AS max_records_per_key
    FROM key_counts
""", [str(USA_FILE)]).df()

display(duplicate_key_check.T)

,0
duplicated_key_groups,200639614.0
excess_records,201102034.0
max_records_per_key,14.0


### 4.1 File conversion due to data storage limits

Due to the large size of the original USA CSV file (43.65 GB), the dataset was converted to Parquet format to improve storage efficiency and facilitate subsequent data processing. The following sections describe the conversion and validate the consistency of the converted data.

In [31]:
USA_PARQUET = PROCESSED_DIR / "usa_trade_monthly.parquet"

con.execute("SET memory_limit = '8GiB'")
con.execute("SET threads = 4")
con.execute("SET preserve_insertion_order = false")

usa_file_sql = USA_FILE.as_posix().replace("'", "''")
usa_parquet_sql = USA_PARQUET.as_posix().replace("'", "''")

con.execute(f"""
    COPY (
        SELECT
            month_id,
            state_id,
            state_name,
            trade_flow_name,
            country_id,
            country_name,
            product_id,
            product_name,
            trade_value
        FROM read_csv_auto(
            '{usa_file_sql}',
            sample_size = 100000
        )
    )
    TO '{usa_parquet_sql}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Parquet file created: {USA_PARQUET}")

Parquet file created: c:\Transfer\InterdisciplinaryProject\data\processed\usa_trade_monthly.parquet


In [32]:
usa_validation = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT month_id) AS distinct_months,
        MIN(month_id) AS first_month,
        MAX(month_id) AS last_month,
        COUNT(DISTINCT state_id) AS distinct_states,
        COUNT(DISTINCT product_id) AS distinct_products,
        COUNT(DISTINCT country_id) AS distinct_partners,
        COUNT(DISTINCT trade_flow_name) AS distinct_trade_flows
    FROM read_parquet(?)
""", [str(USA_PARQUET)]).df()

display(usa_validation)

,total_rows,distinct_months,first_month,last_month,distinct_states,distinct_products,distinct_partners,distinct_trade_flows
0,401741648,112,201701,202604,54,5784,231,2


In [37]:
duplicate_key_summary = con.execute("""
    SELECT
        COUNT(*) AS duplicated_key_groups,
        SUM(record_count - 1) AS excess_records,
        MAX(record_count) AS max_records_per_key
    FROM (
        SELECT COUNT(*) AS record_count
        FROM read_parquet(?)
        GROUP BY
            month_id,
            state_id,
            product_id,
            country_id,
            trade_flow_name
        HAVING COUNT(*) > 1
    ) AS duplicates
""", [str(USA_PARQUET)]).df()

display(duplicate_key_summary)

,duplicated_key_groups,excess_records,max_records_per_key
0,200639614,201102034.0,14


### 4.2 Conflicting trade value check

This query identifies analytical key groups containing more than one distinct `trade_value` to detect potential inconsistencies.

In [38]:
conflicting_keys = con.execute("""
    SELECT COUNT(*) AS conflicting_key_groups
    FROM (
        SELECT
            month_id,
            state_id,
            product_id,
            country_id,
            trade_flow_name
        FROM read_parquet(?)
        GROUP BY
            month_id,
            state_id,
            product_id,
            country_id,
            trade_flow_name
        HAVING COUNT(DISTINCT trade_value) > 1
    ) AS conflicts
""", [str(USA_PARQUET)]).df()

display(conflicting_keys)

,conflicting_key_groups
0,152345


### 4.3 Inspection of conflicting trade values

This query displays records for 20 analytical keys with conflicting `trade_value` values to examine the differences between records sharing the same key.

In [39]:
conflicting_value_sample = con.execute("""
    WITH conflicting_keys AS (
        SELECT
            month_id,
            state_id,
            product_id,
            country_id,
            trade_flow_name
        FROM read_parquet(?)
        GROUP BY
            month_id,
            state_id,
            product_id,
            country_id,
            trade_flow_name
        HAVING COUNT(DISTINCT trade_value) > 1
        LIMIT 20
    )
    SELECT
        p.month_id,
        p.state_id,
        p.state_name,
        p.product_id,
        p.product_name,
        p.country_id,
        p.country_name,
        p.trade_flow_name,
        p.trade_value
    FROM read_parquet(?) AS p
    INNER JOIN conflicting_keys AS c
        ON p.month_id = c.month_id
        AND p.state_id = c.state_id
        AND p.product_id = c.product_id
        AND p.country_id = c.country_id
        AND p.trade_flow_name = c.trade_flow_name
    ORDER BY
        p.month_id,
        p.state_id,
        p.product_id,
        p.country_id,
        p.trade_flow_name,
        p.trade_value
""", [
    str(USA_PARQUET),
    str(USA_PARQUET)
]).df()

display(conflicting_value_sample)

,month_id,state_id,state_name,product_id,product_name,country_id,country_name,trade_flow_name,trade_value
0,201708,04000US06,California,999999,Commodities not specified according to kind,gbr,United Kingdom,Imports,16637
1,201708,04000US06,California,999999,Commodities not specified according to kind,gbr,United Kingdom,Imports,16637
2,201708,04000US06,California,999999,Commodities not specified according to kind,gbr,United Kingdom,Imports,25902
3,201708,04000US06,California,999999,Commodities not specified according to kind,gbr,United Kingdom,Imports,25902
4,201708,04000US06,California,999999,Commodities not specified according to kind,gbr,United Kingdom,Imports,44171
...,...,...,...,...,...,...,...,...,...
87,202502,04000US37,North Carolina,999999,Commodities not specified according to kind,jor,Jordan,Exports,2894
88,202503,04000US27,Minnesota,999999,Commodities not specified according to kind,nzl,New Zealand,Imports,0
89,202503,04000US27,Minnesota,999999,Commodities not specified according to kind,nzl,New Zealand,Imports,0
90,202503,04000US27,Minnesota,999999,Commodities not specified according to kind,nzl,New Zealand,Imports,33095


### 4.4 Descriptive field consistency

This query checks whether records sharing the same analytical key also have consistent state, country, and product names, helping distinguish trade-value conflicts from inconsistencies in descriptive fields.

Note: SAMPLE-BASED ANALYSIS*

In [50]:
key_columns = [
    "month_id",
    "state_id",
    "product_id",
    "country_id",
    "trade_flow_name"
]

name_columns = [
    "state_name",
    "product_name",
    "country_name"
]

# Use the conflicting-key sample already obtained
sample_consistency = (
    conflicting_value_sample
    .groupby(key_columns, dropna=False)
    .agg(
        rows=("trade_value", "size"),
        distinct_trade_values=("trade_value", "nunique"),
        **{
            f"distinct_{col}": (col, "nunique")
            for col in name_columns
        }
    )
    .reset_index()
)

display(sample_consistency)

,month_id,state_id,product_id,country_id,trade_flow_name,rows,distinct_trade_values,distinct_state_name,distinct_product_name,distinct_country_name
0,201708,04000US06,999999,gbr,Imports,12,6,1,1,1
1,201709,04000US55,880330,mys,Exports,4,2,1,1,1
2,201710,04000US13,880330,dom,Exports,4,2,1,1,1
3,201711,04000US39,880330,sgp,Exports,4,2,1,1,1
4,201712,04000US18,880330,hun,Exports,4,2,1,1,1
5,201807,04000US05,999999,gbr,Imports,4,2,1,1,1
6,201808,04000US48,820320,cod,Exports,4,2,1,1,1
7,201811,04000US29,999999,mys,Imports,4,2,1,1,1
8,201811,04000US48,999999,pol,Exports,4,2,1,1,1
9,201812,04000US19,999999,irl,Imports,4,2,1,1,1


### 4.5 Sample-based analysis of conflicting trade values

Classify sampled analytical keys by whether they contain zero values, one distinct positive trade value, or multiple distinct positive trade values.

In [51]:
key_columns = [
    "month_id",
    "state_id",
    "product_id",
    "country_id",
    "trade_flow_name"
]

# Summarize trade values within each sampled key
value_patterns = (
    conflicting_value_sample
    .groupby(key_columns, dropna=False)
    .agg(
        rows=("trade_value", "size"),
        has_zero=("trade_value", lambda x: (x == 0).any()),
        distinct_positive_values=(
            "trade_value",
            lambda x: x[x > 0].nunique()
        ),
        distinct_all_values=("trade_value", "nunique")
    )
    .reset_index()
)

# Classify each key
value_patterns["pattern"] = np.select(
    [
        value_patterns["has_zero"]
        & (value_patterns["distinct_positive_values"] == 1),

        value_patterns["distinct_positive_values"] > 1,

        value_patterns["has_zero"]
        & (value_patterns["distinct_positive_values"] == 0)
    ],
    [
        "Zero + one distinct positive value",
        "Multiple distinct positive values",
        "Zero values only"
    ],
    default="Positive values only; one distinct positive value"
)

# Count keys in each pattern
pattern_summary = (
    value_patterns["pattern"]
    .value_counts()
    .rename_axis("pattern")
    .reset_index(name="sampled_key_groups")
)

display(pattern_summary)
display(value_patterns)

,pattern,sampled_key_groups
0,Zero + one distinct positive value,13
1,Multiple distinct positive values,7


,month_id,state_id,product_id,country_id,trade_flow_name,rows,has_zero,distinct_positive_values,distinct_all_values,pattern
0,201708,04000US06,999999,gbr,Imports,12,False,6,6,Multiple distinct positive values
1,201709,04000US55,880330,mys,Exports,4,False,2,2,Multiple distinct positive values
2,201710,04000US13,880330,dom,Exports,4,True,1,2,Zero + one distinct positive value
3,201711,04000US39,880330,sgp,Exports,4,True,1,2,Zero + one distinct positive value
4,201712,04000US18,880330,hun,Exports,4,True,1,2,Zero + one distinct positive value
5,201807,04000US05,999999,gbr,Imports,4,True,1,2,Zero + one distinct positive value
6,201808,04000US48,820320,cod,Exports,4,True,1,2,Zero + one distinct positive value
7,201811,04000US29,999999,mys,Imports,4,True,1,2,Zero + one distinct positive value
8,201811,04000US48,999999,pol,Exports,4,True,1,2,Zero + one distinct positive value
9,201812,04000US19,999999,irl,Imports,4,True,1,2,Zero + one distinct positive value


### 4.6 Remove exact duplicate records

#### Configure DuckDB for large-scale deduplication

Reduce parallel processing overhead and allow more temporary disk space for deduplicating the large USA dataset.

In [8]:

con.execute("SET threads = 2;")
con.execute("SET preserve_insertion_order = false;")
con.execute("SET max_temp_directory_size = '28GiB';")

#### Partition the USA dataset for deduplication

Distribute records into 16 temporary Parquet partitions using a hash of all source columns, ensuring exact duplicates are assigned to the same partition for independent deduplication.

In [ ]:
source_file = "../Data/processed/usa_trade_monthly.parquet"
partition_dir = Path("../Data/processed/usa_dedup_partitions")

partition_dir.mkdir(parents=True, exist_ok=True)

con.execute(f"""
    COPY (
        SELECT
            *,
            hash(
                month_id, state_id, state_name, trade_flow_name,
                country_id, country_name, product_id, product_name,
                trade_value
            ) % 16 AS partition_id
        FROM read_parquet('{source_file}')
    )
    TO '{partition_dir.as_posix()}'
    (FORMAT PARQUET, PARTITION_BY (partition_id), COMPRESSION ZSTD);
""")


#### Individual partition deduplication

Deduplicate each hash partition independently across the original nine source columns, preserving records with different trade values.

In [14]:
from pathlib import Path

project_dir = Path.cwd().parent
partition_dir = project_dir / "Data" / "processed" / "usa_dedup_partitions"
deduplicated_dir = project_dir / "Data" / "processed" / "usa_deduplicated_partitions"

deduplicated_dir.mkdir(parents=True, exist_ok=True)

partition_files = sorted(partition_dir.glob("partition_id=*/*.parquet"))

print(f"Partitions found: {len(partition_files)}")

for i, file in enumerate(partition_files, start=1):
    output_file = deduplicated_dir / f"deduplicated_{i:02d}.parquet"

    con.execute(f"""
        COPY (
            SELECT DISTINCT
                month_id,
                state_id,
                state_name,
                trade_flow_name,
                country_id,
                country_name,
                product_id,
                product_name,
                trade_value
            FROM read_parquet('{file.as_posix()}')
        )
        TO '{output_file.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD);
    """)

    print(f"Completed partition {i}/{len(partition_files)}")

Partitions found: 16
Completed partition 1/16
Completed partition 2/16
Completed partition 3/16
Completed partition 4/16
Completed partition 5/16
Completed partition 6/16
Completed partition 7/16
Completed partition 8/16
Completed partition 9/16
Completed partition 10/16
Completed partition 11/16
Completed partition 12/16
Completed partition 13/16
Completed partition 14/16
Completed partition 15/16
Completed partition 16/16


#### Validate deduplicated partitions

Confirm that all 16 deduplicated partition files exist and calculate their combined row count before aggregating trade values.

In [15]:
from pathlib import Path

deduplicated_dir = (
    Path.cwd().parent
    / "Data"
    / "processed"
    / "usa_deduplicated_partitions"
)

deduplicated_files = sorted(deduplicated_dir.glob("deduplicated_*.parquet"))

print(f"Deduplicated files found: {len(deduplicated_files)}")

deduplicated_rows = con.execute("""
    SELECT COUNT(*)
    FROM read_parquet(?)
""", [[str(file) for file in deduplicated_files]]).fetchone()[0]

print(f"Total rows after deduplication: {deduplicated_rows:,}")

Deduplicated files found: 16
Total rows after deduplication: 200,827,783


### 4.7 Aggregate trade values across states

Sum `trade_value` by month, product, partner country, and trade flow, excluding state identifiers to produce a country-level monthly dataset comparable with the other datasets.

In [ ]:
project_dir = Path.cwd().parent
deduplicated_dir = project_dir / "Data" / "processed" / "usa_deduplicated_partitions"
output_file = project_dir / "Data" / "processed" / "usa_trade_aggregated.parquet"

deduplicated_files = sorted(deduplicated_dir.glob("deduplicated_*.parquet"))

con.execute(f"""
    COPY (
        SELECT
            month_id,
            country_id,
            country_name,
            product_id,
            product_name,
            trade_flow_name,
            SUM(trade_value) AS trade_value
        FROM read_parquet({[str(f) for f in deduplicated_files]})
        GROUP BY
            month_id,
            country_id,
            country_name,
            product_id,
            product_name,
            trade_flow_name
    )
    TO '{output_file.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

print(f"Aggregated file created: {output_file}")

Aggregated file created: c:\Transfer\InterdisciplinaryProject\Data\processed\usa_trade_aggregated.parquet


## 5.Post-Processing Validation

### 5.1 Validate the aggregated dataset.
Check row count, temporal coverage, number of products and partner countries, missing values, and uniqueness of the analytical key.

In [ ]:
project_dir = Path.cwd().parent
aggregated_file = project_dir / "Data" / "processed" / "usa_trade_aggregated.parquet"

validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT month_id) AS unique_months,
        MIN(month_id) AS first_month,
        MAX(month_id) AS last_month,
        COUNT(DISTINCT product_id) AS unique_products,
        COUNT(DISTINCT country_id) AS unique_partners,
        COUNT(DISTINCT trade_flow_name) AS unique_trade_flows,
        COUNT(*) FILTER (WHERE month_id IS NULL) AS missing_month_id,
        COUNT(*) FILTER (WHERE product_id IS NULL) AS missing_product_id,
        COUNT(*) FILTER (WHERE country_id IS NULL) AS missing_country_id,
        COUNT(*) FILTER (WHERE trade_flow_name IS NULL) AS missing_trade_flow,
        COUNT(*) FILTER (WHERE trade_value IS NULL) AS missing_trade_value
    FROM read_parquet('{aggregated_file.as_posix()}')
""").fetchdf()

display(validation)

,total_rows,unique_months,first_month,last_month,unique_products,unique_partners,unique_trade_flows,missing_month_id,missing_product_id,missing_country_id,missing_trade_flow,missing_trade_value
0,38476493,112,201701,202604,5784,231,2,0,0,0,0,0


### 5.2 Validate analytical key uniqueness

Check whether any duplicate combinations remain for month, partner country, product, and trade flow in the aggregated dataset.

In [18]:
duplicate_check = con.execute("""
    SELECT COUNT(*) AS duplicate_key_groups
    FROM (
        SELECT
            month_id,
            country_id,
            product_id,
            trade_flow_name
        FROM read_parquet('../Data/processed/usa_trade_aggregated.parquet')
        GROUP BY
            month_id,
            country_id,
            product_id,
            trade_flow_name
        HAVING COUNT(*) > 1
    )
""").fetchdf()

display(duplicate_check)

,duplicate_key_groups
0,0


### 5.3 Validate trade-value totals

Compare the total trade value before deduplication, after deduplication, and after aggregation to quantify the effect of removing exact duplicate records and confirm that aggregation preserves the deduplicated total.

In [19]:
from pathlib import Path

project_dir = Path.cwd().parent
source_file = project_dir / "Data" / "processed" / "usa_trade_monthly.parquet"
deduplicated_dir = project_dir / "Data" / "processed" / "usa_deduplicated_partitions"
aggregated_file = project_dir / "Data" / "processed" / "usa_trade_aggregated.parquet"

deduplicated_files = sorted(deduplicated_dir.glob("deduplicated_*.parquet"))

totals = con.execute("""
    SELECT 'Original' AS stage,
           COUNT(*) AS rows,
           SUM(trade_value) AS total_trade_value
    FROM read_parquet(?)

    UNION ALL

    SELECT 'After deduplication',
           COUNT(*),
           SUM(trade_value)
    FROM read_parquet(?)

    UNION ALL

    SELECT 'After aggregation',
           COUNT(*),
           SUM(trade_value)
    FROM read_parquet(?)
""", [
    str(source_file),
    [str(f) for f in deduplicated_files],
    str(aggregated_file)
]).fetchdf()

display(totals)

,stage,rows,total_trade_value
0,Original,401741648,8.768300e+13
1,After deduplication,200827783,4.384150e+13
2,After aggregation,38476493,4.384150e+13


###  Remove temporary intermediate files

Delete the temporary hash partitions and deduplicated partition files after validating the aggregated output, while retaining the original source data and final processed dataset.

In [20]:
from pathlib import Path
import shutil

project_dir = Path.cwd().parent
processed_dir = project_dir / "Data" / "processed"

intermediate_dirs = [
    processed_dir / "usa_dedup_partitions",
    processed_dir / "usa_deduplicated_partitions",
]

for directory in intermediate_dirs:
    if directory.exists():
        shutil.rmtree(directory)
        print(f"Deleted: {directory}")
    else:
        print(f"Already absent: {directory}")

Deleted: c:\Transfer\InterdisciplinaryProject\Data\processed\usa_dedup_partitions
Deleted: c:\Transfer\InterdisciplinaryProject\Data\processed\usa_deduplicated_partitions


# USA Trade Data Preprocessing Report

## 1. Objective

The objective of this preprocessing workflow was to inspect, clean, consolidate, and validate the USA international trade dataset before using it for exploratory data analysis and forecasting.

The workflow included data inspection, missing-value assessment, duplicate investigation, exact duplicate removal, aggregation of trade values, and post-processing validation. The resulting dataset retains monthly trade flows by product, partner country, and trade-flow direction, without state-level granularity.

## 2. Data Source and Input File(s)

### 2.1 Source data

The dataset was obtained from the USA state-level international trade data provided by the U.S. Census Bureau.

**Original input file:** `trade_s_usa_state_m_hs.csv`

The original CSV file was approximately 43.65 GB. Due to its size, it was converted to Parquet format to improve storage efficiency and facilitate subsequent analytical processing.

### 2.2 File conversion

The converted dataset was saved as:

`data/processed/usa_trade_monthly.parquet`

The Parquet file was approximately 2.10 GiB.

The converted dataset contained 401,741,648 rows, matching the number of observations in the original dataset.

### 2.3 Variables

The source dataset contained the following nine columns:

- `month_id`
- `state_id`
- `state_name`
- `trade_flow_name`
- `country_id`
- `country_name`
- `product_id`
- `product_name`
- `trade_value`

The dataset contains monthly trade values for imports and exports, disaggregated by state, product, and partner country.

## 3. Data Inspection and Quality Assessment

### 3.1 Initial data overview

The initial inspection identified the following characteristics:

| Attribute | Result |
|---|---:|
| Total rows | 401,741,648 |
| Distinct months | 112 |
| First month | 201701 |
| Last month | 202604 |
| Distinct states | 54 |
| Distinct products | 5,784 |
| Distinct partner countries | 231 |
| Distinct trade flows | 2 |

The temporal coverage extends from January 2017 to April 2026.

### 3.2 Temporal continuity

The expected monthly sequence from January 2017 through April 2026 contains 112 months. The dataset contained all 112 expected months, with no missing or unexpected months.

### 3.3 Initial data completeness

The following analytical variables were examined for missing values:

- `month_id`
- `state_id`
- `product_id`
- `country_id`
- `trade_flow_name`
- `trade_value`

No missing values were detected in these variables across the 401,741,648 source observations.

The descriptive variables `state_name`, `country_name`, and `product_name` were not included in this initial null check.

## 4. Duplicate Investigation

### 4.1 Proposed analytical key

The initial analytical key was defined as:

`month_id + state_id + product_id + country_id + trade_flow_name`

This key represents a monthly trade observation for a particular state, product, partner country, and trade-flow direction.

### 4.2 Duplicate key investigation

The initial duplicate-key investigation identified:

| Metric | Result |
|---|---:|
| Duplicated analytical key groups | 200,639,614 |
| Excess records across duplicated key groups | 201,102,034 |
| Maximum records per key | 14 |

The existence of repeated analytical keys did not necessarily imply that all associated records were identical. Some groups contained different trade values.

### 4.3 Conflicting trade values

A total of 152,345 analytical key groups were identified as having more than one distinct `trade_value`.

A sample of 20 conflicting key groups was examined to investigate the structure of these records. Within this sample:

- 13 groups contained zero and one distinct positive trade value.
- 7 groups contained multiple distinct positive trade values.

The sampled records showed consistent state, product, and country names within each examined analytical key.

Because computational limitations prevented a comprehensive consistency check across all conflicting groups, the findings from this sample were treated as a working assumption rather than a verified property of the entire dataset.

## 5. Data Cleaning and Consolidation

### 5.1 Conversion to Parquet

The original CSV was converted to Parquet format to reduce storage requirements and facilitate analytical processing.

The converted dataset retained all 401,741,648 source observations.

### 5.2 Exact duplicate removal

Due to the size of the dataset, a single full-table `SELECT DISTINCT` operation exceeded DuckDB's temporary-storage limit.

To complete the operation, the dataset was partitioned into 16 temporary Parquet partitions using a hash of all nine source columns. Identical records were assigned to the same partition, allowing exact duplicate removal to be performed independently within each partition.

The deduplication process produced the following results:

| Metric | Result |
|---|---:|
| Rows before deduplication | 401,741,648 |
| Rows after deduplication | 200,827,783 |
| Exact duplicate excess rows removed | 200,913,865 |

Only rows identical across all nine source columns were removed. Records with different trade values were retained before aggregation.

### 5.3 Aggregation of trade values

Following the approach used for the South Korea dataset, `trade_value` was summed across records sharing the same analytical key.

State-level detail was removed because the forecasting analysis does not require separate state-level series. Consequently, trade values were aggregated across states using the following key:

`month_id + country_id + product_id + trade_flow_name`

The descriptive variables `country_name` and `product_name` were retained in the output.

No weight aggregation was performed because the USA source dataset does not contain a weight variable.

The aggregated dataset was saved as:

`data/processed/usa_trade_aggregated.parquet`

### 5.4 Resource management

The partitioned approach was adopted because full-table deduplication exceeded the available temporary-storage limit. Once the aggregated output had been validated, the temporary hash partitions and deduplicated partition files were removed to reclaim disk space.

The original raw CSV, converted source Parquet file, final aggregated Parquet file, and preprocessing notebook were retained.

## 6. Post-Processing Validation

### 6.1 Dataset overview

The final aggregated dataset contained the following characteristics:

| Attribute | Result |
|---|---:|
| Total rows | 38,476,493 |
| Distinct months | 112 |
| First month | 201701 |
| Last month | 202604 |
| Distinct products | 5,784 |
| Distinct partner countries | 231 |
| Distinct trade flows | 2 |

The final dataset preserves the full observed temporal coverage from January 2017 through April 2026.

### 6.2 Missing-value validation

The final dataset was checked for missing values in:

- `month_id`
- `product_id`
- `country_id`
- `trade_flow_name`
- `trade_value`

No missing values were detected in these variables.

### 6.3 Analytical key uniqueness

The final analytical key was defined as:

`month_id + country_id + product_id + trade_flow_name`

The validation query found **zero duplicated analytical key groups**, confirming that each combination of month, partner country, product, and trade-flow direction occurs at most once in the aggregated dataset.

### 6.4 Trade-value total validation

The total trade value was compared before deduplication, after deduplication, and after aggregation.

| Processing stage | Rows | Total trade value |
|---|---:|---:|
| Original dataset | 401,741,648 | \(8.76830 \times 10^{13}\) |
| After exact duplicate removal | 200,827,783 | \(4.38415 \times 10^{13}\) |
| After aggregation | 38,476,493 | \(4.38415 \times 10^{13}\) |

The total trade value after aggregation matched the total after exact duplicate removal. This confirms that the aggregation operation preserved the sum of the deduplicated records.

However, removing exact duplicates reduced the total trade value by approximately half. This substantial reduction was documented as an important consequence of the deduplication procedure.

## 7. Final Output

The final processed dataset is:

`data/processed/usa_trade_aggregated.parquet`

It contains monthly trade values by product, partner country, and trade-flow direction, aggregated across states.

The dataset covers January 2017 through April 2026 and contains 38,476,493 rows. The analytical key is unique, and no missing values were detected in the checked analytical variables.

The final output is intended for subsequent exploratory data analysis, forecasting-series construction, and model benchmarking.

## 8. Remaining Methodological Considerations

### 8.1 Interpretation of multiple trade values

The preprocessing procedure retains distinct trade-value observations before aggregation. The decision to sum them follows the approach adopted for the South Korea dataset.

However, the interpretation of multiple distinct trade values sharing the same analytical key was not independently verified against the source documentation. The aggregation therefore relies on the working assumption that these values can be combined.

### 8.2 Sample-based consistency assessment

Computational limitations restricted the investigation of conflicting records to a sample of 20 analytical key groups. The consistency of descriptive names observed in this sample was not established for all conflicting records.

### 8.3 Effect of exact duplicate removal

The removal of 200,913,865 excess records substantially reduced the total trade value. Although the resulting dataset passed the structural and aggregation checks, the effect of duplicate removal should be considered when interpreting the final trade totals.

### 8.4 Temporal and series-level completeness

The overall monthly coverage is continuous, but this does not establish that every individual product–country–trade-flow series contains observations for all 112 months. Individual-series completeness, zero-value interpretation, and suitability for forecasting remain topics for subsequent exploratory analysis.

### 8.5 Reproducibility

The original CSV, converted source Parquet file, final aggregated output, and preprocessing notebook were retained. The temporary partition files were deleted after validation. Reproducing the result requires rerunning the partitioning, deduplication, aggregation, and validation steps, which may require substantial temporary disk space.

## 9. Conclusion

The USA trade dataset was converted to Parquet, inspected for missing values and temporal gaps, investigated for duplicate analytical keys, deduplicated using a partitioned procedure, and aggregated across states.

The final dataset contains 38,476,493 observations covering 112 consecutive months from January 2017 to April 2026. It contains 5,784 products, 231 partner countries, and two trade-flow directions. No missing values were detected in the checked analytical variables, and no duplicate analytical keys remain.

The aggregation operation preserved the total trade value after deduplication. Nevertheless, the substantial reduction in trade totals caused by exact duplicate removal and the unverified interpretation of multiple distinct trade values remain important methodological considerations for subsequent forecasting analysis.